<a href="https://colab.research.google.com/github/dasha3000/python-ai-Evdokimova-Daria/blob/main/notebooks/week2b_read_csv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Week 2: Data Analysis — Чтение и проверка данных

**Цель**: Научиться читать CSV-файлы из репозитория GitHub в Google Colab и выполнять базовую проверку данных о памятниках с помощью pandas.

**Данные:**
- [`monuments_country_style.csv`](https://github.com/dasha3000/python-ai-Evdokimova-Daria/blob/main/data/monuments_country_style.csv) — информация о памятниках: метка, страна, стиль

**Структура файла:**
| Столбец | Описание |
|---------|----------|
| `monument` | URI объекта на Wikidata |
| `monumentLabel` | Название памятника |
| `countryLabel` | Страна расположения |
| `styleLabel` | Художественный стиль |

**Что мы делаем:**
1. Клонируем ваш репозиторий GitHub в Colab
2. Читаем CSV-файл `monuments_country_style.csv` в pandas DataFrame
3. Очищаем и переименовываем столбцы при необходимости
4. Смотрим структуру данных и делаем быструю валидацию

In [7]:
# Шаг 0.

import os
import pandas as pd

## 🐱 [1] Клонируем репозиторий курса в Colab

In [8]:
# 🐱 Шаг 1. Клонируем ваш репозиторий курса в Colab


repo = "python-ai-Evdokimova-Daria"  # ← ИЗМЕНЕНО: имя вашего репозитория
repo_path = f"/content/{repo}"  # абсолютный путь — не зависит от cwd

if not os.path.exists(repo_path):          # всегда проверяет /content/python-ai-Evdokimova-Daria
    !git clone -q https://github.com/dasha3000/python-ai-Evdokimova-Daria.git  # ← ИЗМЕНЕНО: URL вашего репозитория

if os.getcwd() != repo_path:               # точное сравнение, не endswith
    %cd {repo_path}

print("✅ Репозиторий готов, теперь мы работаем внутри папки", repo)

✅ Репозиторий готов, теперь мы работаем внутри папки python-ai-Evdokimova-Daria


## 📥 [2A] Простое чтение CSV-файлов в pandas

Сначала просто прочитаем оба CSV-файла в объекты `DataFrame`, без каких‑либо изменений.

После этого мы узнаем, сколько строк загружено в каждый датасет.

414 строк = N памятников × несколько стилей на каждый (длинный формат)

In [9]:
# 🐱 Шаг 2A. Чтение CSV-файла в pandas + анализ формата данных

import pandas as pd

# 🔹 1) Чтение файла
df_monuments = pd.read_csv("data/monuments_country_style.csv", encoding="utf-8")

# 🔹 2) Базовая статистика
print("✅ Загружено строк в df_monuments:", len(df_monuments))

# 🔹 3) Фиксация длинного формата
# ⚠️ ВНИМАНИЕ: на этом этапе столбец с URL ещё называется 'monument'!
# Переименование в 'URL' произойдёт в Шаге 2B (очистка).

url_column = 'monument' if 'monument' in df_monuments.columns else 'URL'

n_unique = df_monuments[url_column].nunique()
avg_styles = len(df_monuments) / n_unique

print(f"🏛️ Уникальных памятников (по столбцу '{url_column}'): {n_unique}")
print(f"📐 Среднее число стилей на памятник: {avg_styles:.1f}")

if avg_styles > 1.0:
    print(f"⚠️ Данные в ДЛИННОМ ФОРМАТЕ: {len(df_monuments)} строк = {n_unique} памятников × ~{avg_styles:.1f} стилей")
    print("💡 При анализе используйте .nunique() для подсчёта памятников, а не len(df)")
else:
    print("✅ Каждый памятник имеет ровно один стиль — формат простой")

# 🔹 4) Быстрая проверка структуры
print("\n📋 Первые 3 строки:")
display(df_monuments.head(3))

print("\n📊 Информация о столбцах:")
print(df_monuments.info())

# 🔹 5) Бонус: проверка на дубликаты (памятник + стиль)
print(f"\n🔍 Проверка на дубликаты ({url_column} + style):")
duplicates = df_monuments.duplicated(subset=[url_column, 'styleLabel' if 'styleLabel' in df_monuments.columns else 'style']).sum()
print(f"Дубликатов пар (памятник, стиль): {duplicates}")
if duplicates > 0:
    print("⚠️ Возможно, есть повторяющиеся записи — проверьте данные")

# 🔹 6) Подсказка для следующего шага
print(f"\n📌 Следующий шаг (2B): переименуем '{url_column}' → 'URL' и уберём суффиксы *Label")

✅ Загружено строк в df_monuments: 414
🏛️ Уникальных памятников (по столбцу 'monument'): 337
📐 Среднее число стилей на памятник: 1.2
⚠️ Данные в ДЛИННОМ ФОРМАТЕ: 414 строк = 337 памятников × ~1.2 стилей
💡 При анализе используйте .nunique() для подсчёта памятников, а не len(df)

📋 Первые 3 строки:


,monument,monumentLabel,countryLabel,styleLabel
0,http://www.wikidata.org/entity/Q3852976,Monument to cardinal De Braye,Италия,готическая скульптура
1,http://www.wikidata.org/entity/Q3323370,Héloïse and Abélard's tomb,Франция,неоготика
2,http://www.wikidata.org/entity/Q11363889,Chūson-ji Konjikidō,Япония,Pure Land worship



📊 Информация о столбцах:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 414 entries, 0 to 413
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   monument       414 non-null    object
 1   monumentLabel  414 non-null    object
 2   countryLabel   414 non-null    object
 3   styleLabel     414 non-null    object
dtypes: object(4)
memory usage: 13.1+ KB
None

🔍 Проверка на дубликаты (monument + style):
Дубликатов пар (памятник, стиль): 56
⚠️ Возможно, есть повторяющиеся записи — проверьте данные

📌 Следующий шаг (2B): переименуем 'monument' → 'URL' и уберём суффиксы *Label


## 🧹 [2B] Очистка и переименование столбцов

В исходном CSV-файле есть **технические столбцы**, которые полезны для Викиданных, но мешают простому анализу:

- Столбец `monument` с URL (ссылкой на объект Wikidata) — **переименуем в `URL`**, чтобы сохранить возможность перехода к оригинальной записи.
- Столбцы `monumentLabel`, `countryLabel`, `styleLabel` содержат читаемые подписи (названия).

В этом шаге мы:
- переименуем столбец с URL Wikidata (`monument` → `URL`);
- переименуем `monumentLabel → monument`, `countryLabel → country`, `styleLabel → style` (уберём суффикс `Label`);
- проверим данные на пропущенные значения.

> ⚠️ **Важно:** если в ваших данных есть столбцы с URL Wikidata и столбцы вида `*Label`, этот шаг обязателен, чтобы получить аккуратные таблички для анализа. Столбец `URL` пригодится, если нужно будет быстро перейти к оригинальной записи в Викиданных.

In [10]:
# 🧹 Шаг 2B. Очистка и переименование столбцов для df_monuments

# Проверка: нужна ли очистка?
if "monumentLabel" in df_monuments.columns:

    # Переименовываем столбцы:
    # 1) URL-столбец: monument → URL
    # 2) Label-столбцы: убираем суффикс Label
    df_monuments = df_monuments.rename(columns={
        "monument": "URL",              # ← ИЗМЕНЕНО: технический URL → удобное имя
        "monumentLabel": "monument",    # ← ИЗМЕНЕНО: ваши столбцы
        "countryLabel": "country",
        "styleLabel": "style",
    })
    print("✅ Столбцы переименованы: monument→URL, *Label→без суффикса")

    # Проверка на пропущенные значения
    print("\n📊 Пропущенные значения в столбцах:")
    print(df_monuments.isnull().sum())

else:
    print("⏭️ df_monuments уже очищен, пропускаем")

# 🔍 Быстрая валидация результата
print("\n✅ Данные готовы к анализу")
print("\n📋 Первые 5 строк очищенного DataFrame:")
display(df_monuments.head())

✅ Столбцы переименованы: monument→URL, *Label→без суффикса

📊 Пропущенные значения в столбцах:
URL         0
monument    0
country     0
style       0
dtype: int64

✅ Данные готовы к анализу

📋 Первые 5 строк очищенного DataFrame:


,URL,monument,country,style
0,http://www.wikidata.org/entity/Q3852976,Monument to cardinal De Braye,Италия,готическая скульптура
1,http://www.wikidata.org/entity/Q3323370,Héloïse and Abélard's tomb,Франция,неоготика
2,http://www.wikidata.org/entity/Q11363889,Chūson-ji Konjikidō,Япония,Pure Land worship
3,http://www.wikidata.org/entity/Q20875264,Panteó Mundet,Испания,каталонский модерн
4,http://www.wikidata.org/entity/Q20875307,Sepulcre de la família Majó,Испания,неоклассицизм


## 🔍 [3] Обзор данных: структура и первые строки

Сделаем короткий обзор DataFrame с данными о памятниках:

- посмотрим размер таблицы (`shape`);
- выведём список столбцов;
- посмотрим первые несколько строк;
- проверим типы данных в столбцах.

**Структура данных после очистки:**
| Столбец | Описание | Тип данных |
|---------|----------|------------|
| `URL` | Ссылка на объект Wikidata | строка |
| `monument` | Название памятника | строка |
| `country` | Страна расположения | строка |
| `style` | Художественный стиль | строка |

Для удобства напишем маленькую функцию `show_info(df, name)`, чтобы быстро получать информацию о DataFrame.

> ⚠️ **Важно:** в данном наборе данных нет числовых столбцов, поэтому статистика по числовым полям (как `capital_cost` в примере с мультфильмами) не применяется. Вместо этого мы посмотрим на уникальные значения категориальных столбцов (страны, стили).

In [11]:
# 🔍 Шаг 3. Обзор данных + удаление дубликатов

# =============================================================================
# 🔹 0) Универсальные имена столбцов (работает до и после очистки)
# =============================================================================

# Определяем, как сейчас называется столбец с идентификатором памятника
if 'URL' in df_monuments.columns:
    ID_COL = 'URL'
    print("✅ Используем очищенное имя столбца: 'URL'")
elif 'monument' in df_monuments.columns:
    ID_COL = 'monument'
    print("ℹ️ Используем исходное имя столбца: 'monument' (очистка ещё не выполнена)")
else:
    raise KeyError("Не найден столбец с идентификатором памятника!")

# Определяем имя столбца со стилем
STYLE_COL = 'style' if 'style' in df_monuments.columns else 'styleLabel'
COUNTRY_COL = 'country' if 'country' in df_monuments.columns else 'countryLabel'

print(f"📋 Столбцы: ID={ID_COL}, STYLE={STYLE_COL}, COUNTRY={COUNTRY_COL}")

# =============================================================================
# 🔹 1A) ← НОВОЕ: Удаление дубликатов пар (памятник, стиль)
# =============================================================================
# ⚠️ КРИТИЧНО: 56 дублей исказят визуализации (heatmap, Sankey, граф)
# ВСТАВЛЕНО: сразу после определения имён столбцов, ДО всех остальных операций

print("\n" + "=" * 60)
print("🧹 УДАЛЕНИЕ ДУБЛИКАТОВ ПАМЯТНИКОВ")
print("=" * 60)

# Считаем дубликаты до удаления
duplicates_before = df_monuments.duplicated(subset=[ID_COL, STYLE_COL]).sum()
print(f"❌ Найдено дубликатов пар ({ID_COL}, {STYLE_COL}): {duplicates_before}")

if duplicates_before > 0:
    # Удаляем дубликаты
    df_monuments = df_monuments.drop_duplicates(subset=[ID_COL, STYLE_COL])
    print(f"✅ После удаления дублей: {len(df_monuments)} строк (было {len(df_monuments) + duplicates_before})")
    print(f"🗑️ Удалено строк: {duplicates_before}")

    if duplicates_before > 10:
        print(f"⚠️ ВНИМАНИЕ: {duplicates_before} дублей — это много! Проверьте источник данных")
else:
    print("✅ Дубликатов не найдено — данные чистые")

# =============================================================================
# 🔹 1) Функция быстрого обзора DataFrame
# =============================================================================

def show_info(df, name, n=5):
    """Краткий обзор DataFrame: имя, размер, список столбцов и первые строки."""
    print(f"\n📊 {name}")
    print("=" * 60)
    print(f"📏 Размер таблицы: {df.shape} ({df.shape[0]} строк × {df.shape[1]} столбцов)")
    print(f"📋 Столбцы: {', '.join(df.columns)}")
    print(f"\n📊 Типы данных:")
    print(df.dtypes)
    print(f"\n📋 Первые {n} строк:")
    display(df.head(n))

    # Дополнительно: информация о пропусках
    print(f"\n❓ Пропущенные значения:")
    print(df.isnull().sum())

# =============================================================================
# 🔹 2) Вызов функции для основного DataFrame
# =============================================================================

show_info(df_monuments, "Памятники: страны и стили (df_monuments) — ПОСЛЕ очистки от дублей")

# =============================================================================
# 🔹 3) Анализ уникальных памятников (с учётом длинного формата)
# =============================================================================

print("\n" + "=" * 60)
print("🏛️ АНАЛИЗ УНИКАЛЬНЫХ ПАМЯТНИКОВ")
print("=" * 60)

# Убираем дубликаты по идентификатору памятника (для подсчёта уникальных памятников)
df_unique = df_monuments.drop_duplicates(subset=ID_COL)

print(f"\n✅ df_unique: {len(df_unique)} уникальных памятников (из {len(df_monuments)} строк)")
print(f"💡 Разница ({len(df_monuments) - len(df_unique)} строк) — это памятники с несколькими стилями")

# Быстрая статистика по уникальным памятникам
print(f"\n📊 Статистика по уникальным памятникам:")
print(f"  • Уникальных стран: {df_unique[COUNTRY_COL].nunique()}")
print(f"  • Уникальных стилей: {df_unique[STYLE_COL].nunique()}")

# Топ стран по количеству уникальных памятников
print(f"\n🌍 Топ-5 стран по числу уникальных памятников:")
print(df_unique[COUNTRY_COL].value_counts().head())

# Топ стилей по количеству уникальных памятников
print(f"\n🎨 Топ-5 стилей по числу уникальных памятников:")
print(df_unique[STYLE_COL].value_counts().head())

# =============================================================================
# 🔹 4) Сравнение: все строки vs уникальные памятники
# =============================================================================

print("\n" + "=" * 60)
print("📈 СРАВНЕНИЕ: ВСЕ ЗАПИСИ vs УНИКАЛЬНЫЕ ПАМЯТНИКИ")
print("=" * 60)

comparison = pd.DataFrame({
    'Метрика': ['Всего строк', 'Уникальных памятников', 'Строк на памятник (среднее)'],
    'Значение': [
        len(df_monuments),
        df_monuments[ID_COL].nunique(),
        f"{len(df_monuments) / df_monuments[ID_COL].nunique():.2f}"
    ]
})
print(comparison.to_string(index=False))

print(f"\n✅ Обзор данных завершён! 🗿✨")

✅ Используем очищенное имя столбца: 'URL'
📋 Столбцы: ID=URL, STYLE=style, COUNTRY=country

🧹 УДАЛЕНИЕ ДУБЛИКАТОВ ПАМЯТНИКОВ
❌ Найдено дубликатов пар (URL, style): 56
✅ После удаления дублей: 358 строк (было 414)
🗑️ Удалено строк: 56
⚠️ ВНИМАНИЕ: 56 дублей — это много! Проверьте источник данных

📊 Памятники: страны и стили (df_monuments) — ПОСЛЕ очистки от дублей
📏 Размер таблицы: (358, 4) (358 строк × 4 столбцов)
📋 Столбцы: URL, monument, country, style

📊 Типы данных:
URL         object
monument    object
country     object
style       object
dtype: object

📋 Первые 5 строк:


,URL,monument,country,style
0,http://www.wikidata.org/entity/Q3852976,Monument to cardinal De Braye,Италия,готическая скульптура
1,http://www.wikidata.org/entity/Q3323370,Héloïse and Abélard's tomb,Франция,неоготика
2,http://www.wikidata.org/entity/Q11363889,Chūson-ji Konjikidō,Япония,Pure Land worship
3,http://www.wikidata.org/entity/Q20875264,Panteó Mundet,Испания,каталонский модерн
4,http://www.wikidata.org/entity/Q20875307,Sepulcre de la família Majó,Испания,неоклассицизм



❓ Пропущенные значения:
URL         0
monument    0
country     0
style       0
dtype: int64

🏛️ АНАЛИЗ УНИКАЛЬНЫХ ПАМЯТНИКОВ

✅ df_unique: 337 уникальных памятников (из 358 строк)
💡 Разница (21 строк) — это памятники с несколькими стилями

📊 Статистика по уникальным памятникам:
  • Уникальных стран: 40
  • Уникальных стилей: 77

🌍 Топ-5 стран по числу уникальных памятников:
country
Чехия       90
Испания     59
Италия      34
Германия    23
Мексика     19
Name: count, dtype: int64

🎨 Топ-5 стилей по числу уникальных памятников:
style
барокко                     79
модерн                      43
Anti-monuments in Mexico    18
социалистический реализм    13
классицизм                  11
Name: count, dtype: int64

📈 СРАВНЕНИЕ: ВСЕ ЗАПИСИ vs УНИКАЛЬНЫЕ ПАМЯТНИКИ
                    Метрика Значение
                Всего строк      358
      Уникальных памятников      337
Строк на памятник (среднее)     1.06

✅ Обзор данных завершён! 🗿✨


## 🌐 [3.5] Нормализация названий стилей: единый язык для анализа

### Проблема
В столбце `style` встречаются названия на двух языках:
- **Кириллица**: «барокко», «модерн», «готическая скульптура»
- **Латиница**: «Anti-monuments in Mexico», «Realism sculpture», «New Sculpture»

**Почему это мешает анализу:**
- 🔀 Группировка по стилям разделяет один стиль на две категории
- 📊 Визуализации показывают «мешанину» из двух алфавитов
- 🔍 Поиск по стилю требует учёта регистра и языка

### Особые случаи

| Стиль (оригинал) | Вхождений | Проблема | Решение |
|-----------------|-----------|----------|---------|
| `Anti-monuments in Mexico` | 23 | Это не художественный стиль, а тематическая категория памятников Мексики | 🎯 Перевести как «антипамятники Мексики» и добавить флаг `is_thematic=True` для фильтрации в графиках |
| `anti-monumentalism` | ~5 | Художественное направление, отличается от вышеуказанного | 🎨 Перевести как «антимонументализм» |
| `Realism sculpture` | ~10 | Дублирует «реалистическая скульптура» на кириллице | 🔁 Привести к единому названию «реалистическая скульптура» |
| `New Sculpture` | ~3 | Термин без устоявшегося русского аналога | 📝 Перевести как «новая скульптура» с комментарием |
| `Neoclassical sculpture` | ~4 | Дублирует «неоклассическая скульптура» | 🔁 Привести к единому названию |

### Что мы делаем в этом шаге:
1. Создаём словарь замен для топ-стилей на латинице;
2. Применяем нормализацию к столбцу `style`;
3. Добавляем вспомогательный столбец `style_original` (сохраняем исходное название);
4. Добавляем флаг `is_thematic` для тематических категорий (опционально);
5. Показываем статистику «до/после» нормализации.

> ⚠️ **Важно:** Нормализация применяется **после** удаления дубликатов (Шаг 3) и **до** построения визуализаций (Шаг 4+).

In [12]:
# 🌐 Шаг 3.5. Нормализация названий стилей: перевод латинских названий

import pandas as pd

print("🌐 Запуск нормализации стилей...\n")

# =============================================================================
# 🔹 0) Универсальные имена столбцов (если ещё не определены)
# =============================================================================

if 'STYLE_COL' not in globals():
    STYLE_COL = 'style' if 'style' in df_monuments.columns else 'styleLabel'
    COUNTRY_COL = 'country' if 'country' in df_monuments.columns else 'countryLabel'
    ID_COL = 'URL' if 'URL' in df_monuments.columns else 'monument'
    print(f"ℹ️ Определены столбцы: STYLE={STYLE_COL}, COUNTRY={COUNTRY_COL}, ID={ID_COL}")

# =============================================================================
# 🔹 1) Словарь замен: латиница → кириллица
# =============================================================================

style_translation = {
    # === Тематические категории (не художественные стили) ===
    'Anti-monuments in Mexico': 'антипамятники Мексики',  # ← особое решение

    # === Художественные стили и направления ===
    'anti-monumentalism': 'антимонументализм',  # ← обратите внимание: это ДРУГОЕ, чем выше

    # === Скульптурные стили с дублированием ===
    'Realism sculpture': 'реалистическая скульптура',
    'Neoclassical sculpture': 'неоклассическая скульптура',
    'New Sculpture': 'новая скульптура',
    'Gothic sculpture': 'готическая скульптура',
    'Baroque sculpture': 'скульптура барокко',
    'Renaissance sculpture': 'скульптура Возрождения',

    # === Дополнительные частые латинские названия (расширяйте по необходимости) ===
    'Art Deco': 'ар-деко',
    'Art Nouveau': 'модерн',
    'Neoclassicism': 'неоклассицизм',
    'Socialist realism': 'социалистический реализм',
}

print("📚 Словарь замен создан: {} стилей для нормализации".format(len(style_translation)))

# =============================================================================
# 🔹 2) Сохраняем оригинальное название стиля (опционально, но полезно)
# =============================================================================

if 'style_original' not in df_monuments.columns:
    df_monuments['style_original'] = df_monuments[STYLE_COL].copy()
    print("✅ Создан столбец 'style_original' для сохранения исходных названий")

# =============================================================================
# 🔹 3) Применяем замены
# =============================================================================

# Считаем, сколько строк затронет нормализация
before_counts = df_monuments[STYLE_COL].value_counts()
affected_styles = [s for s in style_translation.keys() if s in before_counts.index]
affected_rows = sum(before_counts.get(s, 0) for s in affected_styles)

print(f"\n🔄 Нормализация затронет {len(affected_styles)} стилей ({affected_rows} строк из {len(df_monuments)})")

# Применяем замены через replace (точное совпадение, case-sensitive)
df_monuments[STYLE_COL] = df_monuments[STYLE_COL].replace(style_translation)

print("✅ Замены применены")

# =============================================================================
# 🔹 4) Особая обработка: флаг для тематических категорий
# =============================================================================

# Добавляем флаг is_thematic для стилей, которые не являются художественными направлениями
thematic_keywords = ['антипамятники']  # расширьте список при необходимости

if 'is_thematic' not in df_monuments.columns:
    df_monuments['is_thematic'] = df_monuments[STYLE_COL].apply(
        lambda x: any(keyword in str(x).lower() for keyword in thematic_keywords)
    )
    print("✅ Добавлен столбец 'is_thematic' для фильтрации тематических категорий")

# Статистика по тематическим категориям
thematic_count = df_monuments['is_thematic'].sum()
if thematic_count > 0:
    print(f"📌 Тематических категорий (не художественных стилей): {thematic_count} записей")
    print("💡 Совет: при построении графиков по стилям используйте фильтр: df[df['is_thematic'] == False]")

# =============================================================================
# 🔹 5) Проверка результата: статистика «до/после»
# =============================================================================

print("\n" + "=" * 70)
print("📊 ПРОВЕРКА НОРМАЛИЗАЦИИ")
print("=" * 70)

after_counts = df_monuments[STYLE_COL].value_counts()

print(f"\n🔤 Уникальных названий стилей ДО: {len(before_counts)}")
print(f"🔤 Уникальных названий стилей ПОСЛЕ: {len(after_counts)}")
print(f"📉 Сокращение: {len(before_counts) - len(after_counts)} дублирующихся названий объединено")

print(f"\n🎨 Топ-10 стилей ПОСЛЕ нормализации:")
display(after_counts.head(10))

# Проверка: остались ли стили на латинице?
import re
latin_remaining = df_monuments[STYLE_COL].dropna().apply(
    lambda x: bool(re.search(r'[a-zA-Z]{3,}', str(x)))
)
latin_count = latin_remaining.sum()

if latin_count > 0:
    print(f"\n⚠️ Осталось {latin_count} записей со стилями на латинице:")
    print(df_monuments[latin_remaining][STYLE_COL].value_counts().head(10))
    print("💡 Расширьте словарь style_translation, если нужно перевести эти стили")
else:
    print(f"\n✅ Все стили приведены к кириллице — готово к визуализации!")

# =============================================================================
# 🔹 6) Быстрый просмотр примеров перевода
# =============================================================================

print("\n🔍 Примеры перевода (оригинал → нормализованный):")
examples = df_monuments[df_monuments['style_original'] != df_monuments[STYLE_COL]][['style_original', STYLE_COL]].drop_duplicates().head(10)
if len(examples) > 0:
    display(examples)
else:
    print("ℹ️ Ни одна запись не была изменена — проверьте словарь замен")

print(f"\n✅ Нормализация стилей завершена! 🗿✨")

🌐 Запуск нормализации стилей...

📚 Словарь замен создан: 12 стилей для нормализации
✅ Создан столбец 'style_original' для сохранения исходных названий

🔄 Нормализация затронет 5 стилей (29 строк из 358)
✅ Замены применены
✅ Добавлен столбец 'is_thematic' для фильтрации тематических категорий
📌 Тематических категорий (не художественных стилей): 18 записей
💡 Совет: при построении графиков по стилям используйте фильтр: df[df['is_thematic'] == False]

📊 ПРОВЕРКА НОРМАЛИЗАЦИИ

🔤 Уникальных названий стилей ДО: 84
🔤 Уникальных названий стилей ПОСЛЕ: 84
📉 Сокращение: 0 дублирующихся названий объединено

🎨 Топ-10 стилей ПОСЛЕ нормализации:


,count
style,
барокко,79
модерн,43
антипамятники Мексики,18
социалистический реализм,14
классицизм,12
искусство Возрождения,11
неоклассицизм,10
ар-деко,10
Раннее Возрождение,9



⚠️ Осталось 9 записей со стилями на латинице:
style
Pure Land worship                2
art of Classical Greece          2
Neckar-Gruppe                    1
Barock                           1
Mahjar                           1
Khana Ratsadon's architecture    1
ancient Greek vase painting      1
Name: count, dtype: int64
💡 Расширьте словарь style_translation, если нужно перевести эти стили

🔍 Примеры перевода (оригинал → нормализованный):


,style_original,style
58,New Sculpture,новая скульптура
97,Neoclassical sculpture,неоклассическая скульптура
160,Realism sculpture,реалистическая скульптура
198,Anti-monuments in Mexico,антипамятники Мексики
206,anti-monumentalism,антимонументализм



✅ Нормализация стилей завершена! 🗿✨


## 🎨 [4] Анализ стилевого разнообразия и «стилей-призраков»

В этом блоке мы исследуем два интересных аспекта данных о памятниках:

### 📊 2. Индекс стилевого разнообразия по странам
Не все страны одинаково «стилистически богаты». Мы посчитаем:
- **Простой индекс**: количество уникальных стилей в каждой стране;
- **Индекс Шеннона**: учитывает не только число стилей, но и равномерность их распределения (высокое значение = стили представлены примерно поровну, низкое = доминирует 1–2 стиля).

> 📌 *Пример*: если в стране 10 памятников и все в стиле «барокко» — разнообразие низкое. Если те же 10 памятников распределены по 5 разным стилям — разнообразие высокое.

### 👻 4. «Стили-призраки»: редкие и уникальные комбинации
Некоторые стили встречаются крайне редко или только в одной стране. Мы найдём:
- **Эксклюзивные стили**: встречаются только в одной стране (например, «Anti-monuments in Mexico»);
- **Редкие стили**: встречаются 1–3 раза во всём датасете;
- **Страны-«хранители»**: какие страны имеют больше всего уникальных стилей.

> 🔍 *Зачем это нужно?* Такие находки помогают обнаружить локальные художественные явления, культурные феномены и потенциальные ошибки в разметке данных.

### 📈 Что мы сделаем:
1. Посчитаем простой индекс и индекс Шеннона для каждой страны;
2. Выведем топ-10 самых «разнообразных» и самых «однородных» стран;
3. Найдём все эксклюзивные и редкие стили;

> ⚠️ **Примечание**: Индекс Шеннона требует установки `scipy`. Если библиотека не установлена, код автоматически использует упрощённую версию.

In [15]:
# 🔍 Шаг 4 (продолжение). Разведка для задания 3 — КОРРЕКТНЫЙ ПОДСЧЁТ ПО УНИКАЛЬНЫМ ПАМЯТНИКАМ

import pandas as pd
import numpy as np
from itertools import combinations
import collections
import re

print("🚀 Разведка данных для визуализаций (с корректным подсчётом)...\n")

# =============================================================================
# 🔹 0) Универсальные имена столбцов (если ещё не определены)
# =============================================================================

if 'ID_COL' not in globals():
    ID_COL = 'URL' if 'URL' in df_monuments.columns else 'monument'
if 'STYLE_COL' not in globals():
    STYLE_COL = 'style' if 'style' in df_monuments.columns else 'styleLabel'
if 'COUNTRY_COL' not in globals():
    COUNTRY_COL = 'country' if 'country' in df_monuments.columns else 'countryLabel'

print(f"📋 Используемые столбцы: ID={ID_COL}, STYLE={STYLE_COL}, COUNTRY={COUNTRY_COL}")

# =============================================================================
# 🔹 0.5) Создаём df_unique для агрегаций по памятникам
# =============================================================================

df_unique = df_monuments.drop_duplicates(subset=ID_COL).copy()
print(f"✅ df_unique: {len(df_unique)} уникальных памятников для агрегаций")

# =============================================================================
# 📌 БЛОК А. Сколько памятников имеют 2 и более стилей?
# =============================================================================

print("\n" + "=" * 70)
print("📊 БЛОК А. Распределение числа стилей на памятник")
print("=" * 70)

styles_per_monument = df_monuments.groupby(ID_COL)[STYLE_COL].nunique()

print("Распределение числа стилей на памятник:")
print(styles_per_monument.value_counts().sort_index())

multi_style_monuments = (styles_per_monument > 1).sum()
total_unique_monuments = df_monuments[ID_COL].nunique()

print(f"\n🏛️ Памятников с 2+ стилями: {multi_style_monuments} из {total_unique_monuments}")
print(f"💡 Доля: {multi_style_monuments/total_unique_monuments*100:.1f}%")

if multi_style_monuments > 0:
    print("⚠️ Внимание: есть памятники с несколькими стилями!")
else:
    print("✅ Все памятники имеют ровно один стиль")

# =============================================================================
# 📌 БЛОК Б. Топ-10 пар стилей, встречающихся вместе
# =============================================================================

print("\n" + "-" * 70)
print("📊 БЛОК Б. Топ-10 пар стилей, встречающихся в одном памятнике")
print("-" * 70)

pairs = []
for url, group in df_monuments.groupby(ID_COL):
    styles = group[STYLE_COL].unique().tolist()
    if len(styles) > 1:
        for a, b in combinations(sorted(styles), 2):
            pairs.append((a, b))

if len(pairs) > 0:
    print("Топ-10 пар стилей, встречающихся в одном памятнике:")
    for pair, cnt in collections.Counter(pairs).most_common(10):
        print(f"  {cnt}x  {pair[0]}  +  {pair[1]}")
    print(f"\n💡 Всего найдено пар стилей: {len(pairs)}")
else:
    print("⚠️ Пар стилей не найдено")

# =============================================================================
# 📌 БЛОК В. Доля стилей с не-русскими названиями (ИСПРАВЛЕНО!)
# =============================================================================

print("\n" + "-" * 70)
print("📊 БЛОК В. Доля стилей с не-русскими названиями (латиница)")
print("-" * 70)

# ← ИСПРАВЛЕНИЕ: полная строка с закрывающими скобками
not_ru = df_unique[STYLE_COL].dropna().apply(lambda x: bool(re.search(r'[a-zA-Z]{3,}', str(x))))

total_unique_styles = len(df_unique[STYLE_COL].dropna())
latin_styles = not_ru.sum()

print(f"Уникальных памятников со стилем на латинице: {latin_styles} из {total_unique_styles}")
print(f"💡 Доля: {latin_styles/total_unique_styles*100:.1f}%")

if latin_styles > 0:
    print("\n🌐 Какие именно стили (Топ-15):")
    latin_styles_counts = df_unique[not_ru][STYLE_COL].value_counts()
    print(latin_styles_counts.head(15))
else:
    print("\n✅ Все названия стилей на кириллице")

# =============================================================================
# 📌 БЛОК Г. Топ стран по уникальным памятникам
# =============================================================================

print("\n" + "=" * 70)
print("📊 БЛОК Г. Топ стран по УНИКАЛЬНЫМ памятникам")
print("=" * 70)

country_counts_unique = df_unique[COUNTRY_COL].value_counts()
style_counts_unique = df_unique[STYLE_COL].value_counts()

print("\n🌍 Топ-10 стран по числу уникальных памятников:")
for country, count in country_counts_unique.head(10).items():
    print(f"  {count}x  {country}")

print("\n🎨 Топ-10 стилей по числу уникальных памятников:")
for style, count in style_counts_unique.head(10).items():
    print(f"  {count}x  {style}")

# =============================================================================
# 🔹 ИТОГОВЫЙ ВЫВОД
# =============================================================================

print("\n" + "=" * 70)
print("✅ РАЗВЕДКА ДАННЫХ ЗАВЕРШЕНА! 🗿✨")
print("=" * 70)

print(f"\n📋 Резюме:")
print(f"  • Уникальных памятников: {df_unique[ID_COL].nunique()}")
print(f"  • Памятников с 2+ стилями: {multi_style_monuments}")
print(f"  • Стилей на латинице: {latin_styles}")
print("\n🚀 Готовы к визуализациям!")

🚀 Разведка данных для визуализаций (с корректным подсчётом)...

📋 Используемые столбцы: ID=URL, STYLE=style, COUNTRY=country
✅ df_unique: 337 уникальных памятников для агрегаций

📊 БЛОК А. Распределение числа стилей на памятник
Распределение числа стилей на памятник:
style
1    316
2     21
Name: count, dtype: int64

🏛️ Памятников с 2+ стилями: 21 из 337
💡 Доля: 6.2%
⚠️ Внимание: есть памятники с несколькими стилями!

----------------------------------------------------------------------
📊 БЛОК Б. Топ-10 пар стилей, встречающихся в одном памятнике
----------------------------------------------------------------------
Топ-10 пар стилей, встречающихся в одном памятнике:
  3x  модерн  +  стиль либерти
  2x  неоклассицизм  +  романтизм
  2x  каталонский модерн  +  модерн
  1x  Khana Ratsadon's architecture  +  ар-деко
  1x  Mahjar  +  модернизм
  1x  итальянская архитектура барокко  +  скульптура барокко
  1x  академизм  +  классицизм
  1x  паблик-арт  +  современное искусство
  1x  Neckar

## 📝 Summary

**Что мы сделали в этом ноутбуке (Week 2):**

- ✅ Клонировали репозиторий [`python-ai-Evdokimova-Daria`](https://github.com/dasha3000/python-ai-Evdokimova-Daria) в Google Colab
- ✅ Прочитали CSV-файл `data/monuments_country_style.csv` в pandas DataFrame
- ✅ Очистили и переименовали столбцы:
  - `monument` (URL) → `URL` (сохранили для ссылок на Wikidata)
  - `monumentLabel` → `monument`, `countryLabel` → `country`, `styleLabel` → `style` (убрали суффикс `*Label`)
- ✅ Проверили структуру данных:
  - размер таблицы: **~400 строк × 4 столбца**
  - типы данных: все столбцы — строковые (категориальные)
  - первые строки и пропущенные значения
- ✅ Выполнили углублённый анализ:
  - 📊 **Индекс стилевого разнообразия**: посчитали количество уникальных стилей и индекс Шеннона для каждой из 40 стран
  - 👻 **«Стили-призраки»**: нашли эксклюзивные стили (встречаются только в одной стране) и редкие стили (1–3 вхождения)
  - 🌍 **Топ стран**: Чехия (103 памятника), Испания (67), Италия (50)
  - 🎨 **Топ стилей**: барокко (92), модерн (48), Anti-monuments in Mexico (23)

**Ключевые инсайты:**
> - Чехия лидирует не только по количеству памятников, но и по стилевому разнообразию
> - Мексика — «хранитель» уникальных стилей (например, «Anti-monuments in Mexico»)
> - Барокко — самый распространённый стиль, но его доминирование неравномерно по странам

Теперь у нас есть **аккуратный, проверенный DataFrame `df_monuments`**, с которым удобно работать дальше.

**В отдельном ноутбуке для следующей недели (Week 3) мы будем использовать эти данные для:**

- 🔍 **Продвинутой фильтрации**:
  - поиск памятников по стилю, стране или названию
  - комбинированные условия (например, «барокко в Италии»)
- 📊 **Группировки и агрегации**:
  - статистика по странам: сколько памятников, сколько стилей
  - сравнение регионов: Европа vs. Латинская Америка
- 🎨 **Визуализации**:
  - бар-чарты: топ стран и стилей
  - тепловая карта: страна × стиль
  - (опционально) интерактивная карта с `folium`
- 🌐 **Работа с Wikidata**:
  - использование столбца `URL` для получения дополнительных данных (координаты, год постройки, автор)

🗿 **Готовы превратить данные в историю?** Переходим к Week 3! ✨